# RNAi control comparisons

Supplementary analysis for the Round 2 response to Reviewer 1.

For each of the six RNAi experiments (fed flies, `Volume` in nL):

1. Mean and SEM of each group
2. Test contrasts against the **responder** control and against the **driver** control
3. **Expected additive** vs **actual** consumption of the double heterozygote

DABEST settings match `06_Survival_and_RNAi`: 5000 bootstrap resamples, `random_seed=12345`
(the library defaults used throughout the paper, here stated explicitly for reproducibility).

In [1]:
#| export
import dabest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from esploco.plotTools import setFont
setFont('inter', 10)
plt.rcParams['svg.fonttype'] = 'none'

paperpath  = '/Users/sangyuxu/Library/Mobile Documents/com~apple~CloudDocs/EspressoManu/'
imagepath  = paperpath + 'Figures202604/'
figurepath = imagepath + '/Figure6/'
picklepath = paperpath + 'DataforFigures/Behavior/Data/pickles/'

# DABEST settings, matched across the paper
RESAMPLES   = 5000
RANDOM_SEED = 12345

Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 39.80it/s]

Numba compilation complete!


## Load data

`rnaidf` is written out by `06_Survival_and_RNAi` (Figure 6E cell). `w1118` fed flies are
excluded from `rnaidf` and are loaded separately here as the wild-type reference for the
additive expectation.

In [2]:
rnaidf = pd.read_csv('rnaidf.csv', index_col=0)

# ── w1118 fed reference ────────────────────────────────────────────────────
# Loaded from the Figure 6 metadata if available, else falls back to the
# values reported for Figure 1H.
try:
    metapath = paperpath + 'DataforFigures/Behavior/Data/Metadata/Figure6exptMeta.csv'
    fullMetaData = pd.read_csv(metapath)
    wrow = fullMetaData.loc[fullMetaData['Expt'] == 'w1118']
    w1118 = pd.read_csv(picklepath + wrow.iloc[0, 1])
    w1118['Volume'] = w1118['Volume'] * 1000
    w1118 = w1118.loc[w1118['Starvation'].round() == 0]
    W_VALS = w1118['Volume'].values
    W_MEAN = W_VALS.mean()
    print(f'w1118 fed loaded: n={len(W_VALS)}, mean={W_MEAN:.2f}, '
          f'sem={W_VALS.std(ddof=1)/np.sqrt(len(W_VALS)):.2f}')
except Exception as e:
    W_VALS = None
    W_MEAN = 62.34530666666666
    W_SEM  = 6.354144837739075
    W_N    = 150
    print(f'w1118 raw not loaded ({type(e).__name__}); '
          f'using reported mean={W_MEAN:.2f}, sem={W_SEM:.2f}, n={W_N}')

w1118 fed loaded: n=150, mean=62.35, sem=6.35


In [3]:
# Experiment -> (driver group, test group, display label for the RNAi allele)
EXPTS = {
    'TrhTrhi':   ('Trh-Gal4/W',             'Trh-Gal4>RNAi',             'Trhi'),
    'TrhGluti':  ('Trh-Gal4/W',             'Trh-Gal4>RNAi',             'Glut1i'),
    'TrhSut2i':  ('Trh-Gal4/W',             'Trh-Gal4>RNAi',             'sut2i'),
    'TrhVTrhi':  ('Trh-AND-tsh-Gal4/W',     'Trh-AND-tsh-Gal4>RNAi',     'Trhi'),
    'TrhVGluti': ('Trh-AND-tsh-Gal4/W',     'Trh-AND-tsh-Gal4>RNAi',     'Glut1i'),
    'TrhVSut2i': ('Trh-AND-tsh-Gal4/W',     'Trh-AND-tsh-Gal4>RNAi',     'sut2i'),
}
RESPONDER = 'RNAi/W'

assert set(EXPTS).issubset(set(rnaidf['Expt'].unique())), 'Expt labels do not match rnaidf'

## 1. Group means and SEMs

In [6]:
rows = []
for expt, (driver, test, allele) in EXPTS.items():
    df = rnaidf[rnaidf['Expt'] == expt]
    for role, geno in [('Responder ctrl', RESPONDER), ('Driver ctrl', driver), ('Test', test)]:
        v = df.loc[df['GenoReduced1'] == geno, 'Volume'].values
        rows.append({
            'Expt': expt, 'Role': role, 'Genotype': geno, 'Allele': allele,
            'n': len(v),
            'mean': v.mean(),
            'sem': v.std(ddof=1) / np.sqrt(len(v)),
        })

group_summary = pd.DataFrame(rows)
group_summary.to_csv(figurepath + 'rnai_group_summary.csv', index=False)
group_summary.round(2)

,Expt,Role,Genotype,Allele,n,mean,sem
0,TrhTrhi,Responder ctrl,RNAi/W,Trhi,30,55.68,11.98
1,TrhTrhi,Driver ctrl,Trh-Gal4/W,Trhi,30,130.99,13.71
2,TrhTrhi,Test,Trh-Gal4>RNAi,Trhi,29,178.62,12.51
3,TrhGluti,Responder ctrl,RNAi/W,Glut1i,25,77.35,15.07
4,TrhGluti,Driver ctrl,Trh-Gal4/W,Glut1i,29,199.95,17.69
5,TrhGluti,Test,Trh-Gal4>RNAi,Glut1i,35,120.15,18.98
6,TrhSut2i,Responder ctrl,RNAi/W,sut2i,15,25.73,9.87
7,TrhSut2i,Driver ctrl,Trh-Gal4/W,sut2i,16,361.20,51.45
8,TrhSut2i,Test,Trh-Gal4>RNAi,sut2i,28,141.26,23.38
9,TrhVTrhi,Responder ctrl,RNAi/W,Trhi,29,100.18,16.33


## 2. Responder and driver control comparisons

The shared-control plots in **Figure S10** take the responder control as baseline, so the
test-vs-driver contrast is not displayed. Both are computed here as separate two-group
estimations.

In [7]:
def two_group(df, ctrl, test):
    """Two-group mean difference with the paper's DABEST settings."""
    c = dabest.load(df, x='GenoReduced1', y='Volume', idx=(ctrl, test),
                    resamples=RESAMPLES, random_seed=RANDOM_SEED)
    r = c.mean_diff.results.iloc[0]
    return c, {
        'difference': r['difference'],
        'bca_low':    r['bca_low'],
        'bca_high':   r['bca_high'],
    }

contrast_rows = []
contrast_objs = {}
for expt, (driver, test, allele) in EXPTS.items():
    df = rnaidf[rnaidf['Expt'] == expt]
    for label, ctrl in [('vs responder ctrl', RESPONDER), ('vs driver ctrl', driver)]:
        c, res = two_group(df, ctrl, test)
        contrast_objs[(expt, label)] = c
        contrast_rows.append({'Expt': expt, 'Allele': allele, 'Comparison': label,
                              'Control': ctrl, **res})

contrasts = pd.DataFrame(contrast_rows)
contrasts.to_csv(figurepath + 'rnai_control_contrasts.csv', index=False)
contrasts.round(2)

,Expt,Allele,Comparison,Control,difference,bca_low,bca_high
0,TrhTrhi,Trhi,vs responder ctrl,RNAi/W,122.94,89.08,154.01
1,TrhTrhi,Trhi,vs driver ctrl,Trh-Gal4/W,47.63,11.94,83.20
2,TrhGluti,Glut1i,vs responder ctrl,RNAi/W,42.80,-3.89,91.62
3,TrhGluti,Glut1i,vs driver ctrl,Trh-Gal4/W,-79.81,-128.84,-26.01
4,TrhSut2i,sut2i,vs responder ctrl,RNAi/W,115.53,73.33,171.85
5,TrhSut2i,sut2i,vs driver ctrl,Trh-Gal4/W,-219.94,-345.00,-122.76
6,TrhVTrhi,Trhi,vs responder ctrl,RNAi/W,110.42,59.00,160.09
7,TrhVTrhi,Trhi,vs driver ctrl,Trh-AND-tsh-Gal4/W,18.58,-60.63,78.79
8,TrhVGluti,Glut1i,vs responder ctrl,RNAi/W,97.35,47.75,144.08
9,TrhVGluti,Glut1i,vs driver ctrl,Trh-AND-tsh-Gal4/W,-19.12,-88.32,46.06


In [11]:
# Side-by-side view: does the inference depend on which control is used?
wide = contrasts.pivot_table(index=['Expt', 'Allele'], columns='Comparison',
                             values=['difference', 'bca_low', 'bca_high'])
wide = wide.swaplevel(axis=1).sort_index(axis=1)
wide.round(1)

Comparison       vs driver ctrl                    vs responder ctrl          \
                       bca_high bca_low difference          bca_high bca_low   
Expt      Allele                                                               
TrhGluti  Glut1i          -26.0  -128.8      -79.8              91.6    -3.9   
TrhSut2i  sut2i          -122.8  -345.0     -219.9             171.9    73.3   
TrhTrhi   Trhi             83.2    11.9       47.6             154.0    89.1   
TrhVGluti Glut1i           46.1   -88.3      -19.1             144.1    47.8   
TrhVSut2i sut2i            48.9   -13.0       18.4             123.9    63.6   
TrhVTrhi  Trhi             78.8   -60.6       18.6             160.1    59.0   

Comparison                   
                 difference  
Expt      Allele             
TrhGluti  Glut1i       42.8  
TrhSut2i  sut2i       115.5  
TrhTrhi   Trhi        122.9  
TrhVGluti Glut1i       97.3  
TrhVSut2i sut2i        97.5  
TrhVTrhi  Trhi        110.4

## 3. Expected additive vs actual

If each transgene contributes an additive deviation from a common background, the double
heterozygote is expected at

$$E[\text{test}] = \bar{x}_{\text{driver}} + \bar{x}_{\text{responder}} - \bar{x}_{w1118}$$

subtracting $w1118$ once because each parental control already carries that background.
The surplus is $\bar{x}_{\text{test}} - E[\text{test}]$, bootstrapped by resampling all four
groups independently with the same seed and resample count as the DABEST contrasts above.

Computed on two scales:

- **all flies** — the pre-specified endpoint
- **feeders only** — because the groups differ markedly in the proportion of non-feeders
  (see `pct_zero` above), and a mean-scale additive model mixes a propensity difference
  with an intake difference

In [13]:
def additive_surplus(R, D, T, W, resamples=RESAMPLES, seed=RANDOM_SEED):
    """Observed test mean minus the additive expectation, with a percentile CI.

    W may be an array of w1118 values, or a (mean, sem, n) tuple when raw data
    are unavailable, in which case the w1118 mean is resampled parametrically.
    """
    rng = np.random.default_rng(seed)
    if isinstance(W, tuple):
        w_mean, w_sem, _ = W
        w_draws = rng.normal(w_mean, w_sem, resamples)
    else:
        w_draws = np.array([rng.choice(W, len(W), replace=True).mean()
                            for _ in range(resamples)])
        w_mean = np.mean(W)

    draws = np.empty(resamples)
    for i in range(resamples):
        r = rng.choice(R, len(R), replace=True).mean()
        d = rng.choice(D, len(D), replace=True).mean()
        t = rng.choice(T, len(T), replace=True).mean()
        draws[i] = t - (d + r - w_draws[i])

    expected = D.mean() + R.mean() - w_mean
    lo, hi = np.percentile(draws, [2.5, 97.5])
    return {'expected_additive': expected, 'actual': T.mean(),
            'surplus': T.mean() - expected, 'bca_low': lo, 'bca_high': hi}


W_all = W_VALS if W_VALS is not None else (W_MEAN, W_SEM, W_N)

add_rows = []
for expt, (driver, test, allele) in EXPTS.items():
    df = rnaidf[rnaidf['Expt'] == expt]
    g = lambda geno: df.loc[df['GenoReduced1'] == geno, 'Volume'].values
    R, D, T = g(RESPONDER), g(driver), g(test)

    res = additive_surplus(R, D, T, W_all)
    add_rows.append({'Expt': expt, 'Allele': allele, 'Scale': 'all flies', **res})

    if W_VALS is not None:
        Wf = W_VALS[W_VALS > 0]
        res_f = additive_surplus(R[R > 0], D[D > 0], T[T > 0], Wf)
        add_rows.append({'Expt': expt, 'Allele': allele, 'Scale': 'feeders only', **res_f})

additive = pd.DataFrame(add_rows)
additive.to_csv(figurepath + 'rnai_additive_expectation.csv', index=False)
additive.round(2)

,Expt,Allele,Scale,expected_additive,actual,surplus,bca_low,bca_high
0,TrhTrhi,Trhi,all flies,124.33,178.62,54.30,10.16,96.87
1,TrhTrhi,Trhi,feeders only,127.19,178.62,51.43,3.92,96.66
2,TrhGluti,Glut1i,all flies,214.95,120.15,-94.81,-153.21,-36.07
3,TrhGluti,Glut1i,feeders only,195.45,123.68,-71.77,-131.16,-7.35
4,TrhSut2i,sut2i,all flies,324.59,141.26,-183.32,-294.94,-74.95
5,TrhSut2i,sut2i,feeders only,297.82,158.21,-139.60,-252.53,-30.02
6,TrhVTrhi,Trhi,all flies,229.86,210.60,-19.26,-98.88,57.24
7,TrhVTrhi,Trhi,feeders only,226.15,210.60,-15.56,-98.15,59.11
8,TrhVGluti,Glut1i,all flies,141.07,140.82,-0.25,-72.15,73.38
9,TrhVGluti,Glut1i,feeders only,152.05,151.66,-0.39,-79.67,73.07


## Combined table

In [14]:
def fmt(d, lo, hi):
    return f'{d:+.1f} [{lo:.0f}, {hi:.0f}]'

out = []
for expt, (driver, test, allele) in EXPTS.items():
    gs = group_summary[group_summary['Expt'] == expt].set_index('Role')
    cv = contrasts[contrasts['Expt'] == expt].set_index('Comparison')
    ad = additive[(additive['Expt'] == expt) & (additive['Scale'] == 'all flies')].iloc[0]
    out.append({
        'Expt': expt,
        'Responder': f"{gs.loc['Responder ctrl','mean']:.1f} ± {gs.loc['Responder ctrl','sem']:.1f}",
        'Driver':    f"{gs.loc['Driver ctrl','mean']:.1f} ± {gs.loc['Driver ctrl','sem']:.1f}",
        'Test':      f"{gs.loc['Test','mean']:.1f} ± {gs.loc['Test','sem']:.1f}",
        'T - responder': fmt(*cv.loc['vs responder ctrl', ['difference','bca_low','bca_high']]),
        'T - driver':    fmt(*cv.loc['vs driver ctrl',    ['difference','bca_low','bca_high']]),
        'Additive exp.': f"{ad['expected_additive']:.1f}",
        'T - additive':  fmt(ad['surplus'], ad['bca_low'], ad['bca_high']),
    })

summary = pd.DataFrame(out)
summary.to_csv(figurepath + 'rnai_summary_for_response.csv', index=False)
summary

,Expt,Responder,Driver,Test,T - responder,T - driver,Additive exp.,T - additive
0,TrhTrhi,55.7 ± 12.0,131.0 ± 13.7,178.6 ± 12.5,"+122.9 [89, 154]","+47.6 [12, 83]",124.3,"+54.3 [10, 97]"
1,TrhGluti,77.3 ± 15.1,200.0 ± 17.7,120.1 ± 19.0,"+42.8 [-4, 92]","-79.8 [-129, -26]",215.0,"-94.8 [-153, -36]"
2,TrhSut2i,25.7 ± 9.9,361.2 ± 51.4,141.3 ± 23.4,"+115.5 [73, 172]","-219.9 [-345, -123]",324.6,"-183.3 [-295, -75]"
3,TrhVTrhi,100.2 ± 16.3,192.0 ± 29.7,210.6 ± 20.6,"+110.4 [59, 160]","+18.6 [-61, 79]",229.9,"-19.3 [-99, 57]"
4,TrhVGluti,43.5 ± 14.3,159.9 ± 28.8,140.8 ± 20.7,"+97.3 [48, 144]","-19.1 [-88, 46]",141.1,"-0.3 [-72, 73]"
5,TrhVSut2i,27.6 ± 10.3,106.8 ± 11.8,125.1 ± 10.8,"+97.5 [64, 124]","+18.4 [-13, 49]",72.1,"+53.1 [12, 91]"


### Notes

- The additive expectation assumes each transgene contributes a fixed additive shift to the
  same quantity. Where the responder control's deficit is mostly a difference in feeding
  propensity rather than intake (high `pct_zero`), that assumption does not hold on the
  pooled-mean scale, and the `feeders only` rows are the more interpretable comparison.
- `w1118` fed flies come from a different cohort than the RNAi crosses, so the additive
  expectation is approximate and is reported as context rather than as a formal null.
- The `vs driver ctrl` column is the conservative comparison reported in the revised text.